In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

GDP_PATH = "/Users/tonytony/Final Project/Data/Raw/gdp.csv"
LIFE_PATH = "/Users/tonytony/Final Project/Data/Raw/life_expectancy.csv"
POP_PATH = "/Users/tonytony/Final Project/Data/Raw/population.csv"
META_PATH = "/Users/tonytony/Final Project/Data/Raw/world_bank_country_metadata.csv"

print("Paths loaded successfully")
print("GDP:", GDP_PATH)
print("Life:", LIFE_PATH)
print("Population:", POP_PATH)
print("Metadata:", META_PATH)


Paths loaded successfully
GDP: /Users/tonytony/Final Project/Data/Raw/gdp.csv
Life: /Users/tonytony/Final Project/Data/Raw/life_expectancy.csv
Population: /Users/tonytony/Final Project/Data/Raw/population.csv
Metadata: /Users/tonytony/Final Project/Data/Raw/world_bank_country_metadata.csv


In [3]:
meta_df = pd.read_csv(META_PATH)

region_map = meta_df[[
    "Country Code",
    "World Bank Region"
]].copy().rename(columns={
    "Country Code": "country_code",
    "World Bank Region": "wb_region"
})

region_map["country_type"] = np.where(
    region_map["wb_region"].astype(str).str.strip().eq("Aggregates"),
    "Aggregate",
    "Country/Territory"
)

print("Metadata shape:", meta_df.shape)
region_map.head()


Metadata shape: (296, 10)


,country_code,wb_region,country_type
0,ABW,Latin America & Caribbean,Country/Territory
1,AFE,Aggregates,Aggregate
2,AFG,"Middle East, North Africa, Afghanistan & Pakistan",Country/Territory
3,AFR,Aggregates,Aggregate
4,AFW,Aggregates,Aggregate


In [4]:
def load_wdi(path):
    df = pd.read_csv(path, skiprows=4)
    df = df.drop(
        columns=[col for col in df.columns if str(col).startswith("Unnamed")],
        errors="ignore"
    )
    year_cols = [col for col in df.columns if str(col).isdigit()]
    df[year_cols] = df[year_cols].apply(pd.to_numeric, errors="coerce")
    return df

def to_long(df, value_name):
    long_df = df.melt(
        id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
        value_vars=[col for col in df.columns if str(col).isdigit()],
        var_name="year",
        value_name=value_name
    ).rename(columns={
        "Country Name": "country_name",
        "Country Code": "country_code",
        "Indicator Name": "indicator_name",
        "Indicator Code": "indicator_code"
    })

    long_df["year"] = pd.to_numeric(long_df["year"], errors="coerce").astype("Int64")
    long_df[value_name] = pd.to_numeric(long_df[value_name], errors="coerce")
    return long_df

def add_region_info(long_df):
    merged = long_df.merge(region_map, on="country_code", how="left")
    merged = merged[merged["country_type"] == "Country/Territory"].copy()
    return merged

print("Helper functions are ready")


Helper functions are ready


In [5]:
gdp_long = add_region_info(to_long(load_wdi(GDP_PATH), "gdp_per_capita_usd"))
life_long = add_region_info(to_long(load_wdi(LIFE_PATH), "life_expectancy_years"))
pop_long = add_region_info(to_long(load_wdi(POP_PATH), "population_total"))

gdp_long = gdp_long[
    (gdp_long["gdp_per_capita_usd"].notna()) &
    (gdp_long["gdp_per_capita_usd"] > 0)
].copy()

life_long = life_long[
    life_long["life_expectancy_years"].notna()
].copy()

pop_long = pop_long[
    (pop_long["population_total"].notna()) &
    (pop_long["population_total"] > 0)
].copy()

print("GDP long shape:", gdp_long.shape)
print("Life long shape:", life_long.shape)
print("Population long shape:", pop_long.shape)


GDP long shape: (11569, 8)
Life long shape: (13854, 8)
Population long shape: (14075, 8)


In [6]:
pop_long = pop_long.sort_values(["country_code", "year"]).copy()
pop_long["prev_year"] = pop_long.groupby("country_code")["year"].shift(1)
pop_long["prev_population"] = pop_long.groupby("country_code")["population_total"].shift(1)

valid_gap = (pop_long["year"] - pop_long["prev_year"]) == 1

pop_long["population_growth_pct"] = np.where(
    valid_gap & (pop_long["prev_population"] > 0),
    (pop_long["population_total"] / pop_long["prev_population"] - 1) * 100,
    np.nan
)

pop_long[[
    "country_name",
    "country_code",
    "year",
    "population_total",
    "population_growth_pct"
]].head(10)


,country_name,country_code,year,population_total,population_growth_pct
0,Aruba,ABW,1960,54922.0,NaN
266,Aruba,ABW,1961,55578.0,1.194421
532,Aruba,ABW,1962,56320.0,1.335061
798,Aruba,ABW,1963,57002.0,1.210938
1064,Aruba,ABW,1964,57619.0,1.082418
1330,Aruba,ABW,1965,58190.0,0.990993
1596,Aruba,ABW,1966,58694.0,0.866128
1862,Aruba,ABW,1967,58990.0,0.504310
2128,Aruba,ABW,1968,59069.0,0.133921
2394,Aruba,ABW,1969,59052.0,-0.028780


In [7]:
panel_events_df = gdp_long[[
    "country_name",
    "country_code",
    "wb_region",
    "year",
    "gdp_per_capita_usd"
]].merge(
    life_long[["country_code", "year", "life_expectancy_years"]],
    on=["country_code", "year"],
    how="inner"
).merge(
    pop_long[["country_code", "year", "population_total", "population_growth_pct"]],
    on=["country_code", "year"],
    how="inner"
)

panel_events_df["log_gdp_per_capita"] = np.log(panel_events_df["gdp_per_capita_usd"])
panel_events_df["log_population_total"] = np.log(panel_events_df["population_total"])

panel_events_df = panel_events_df.sort_values(["country_code", "year"]).copy()
panel_events_df["target_log_gdp_next_year"] = panel_events_df.groupby("country_code")["log_gdp_per_capita"].shift(-1)

print("panel_events_df shape:", panel_events_df.shape)
panel_events_df.head()


panel_events_df shape: (11373, 11)


,country_name,country_code,wb_region,year,gdp_per_capita_usd,life_expectancy_years,population_total,population_growth_pct,log_gdp_per_capita,log_population_total,target_log_gdp_next_year
3665,Aruba,ABW,Latin America & Caribbean,1986,6767.559229,71.831,59931.0,-2.911159,8.819896,11.000949,9.017246
3833,Aruba,ABW,Latin America & Caribbean,1987,8244.045660,72.448,59159.0,-1.288148,9.017246,10.987984,9.215951
4007,Aruba,ABW,Latin America & Caribbean,1988,10056.261393,72.519,59331.0,0.290742,9.215951,10.990887,9.350730
4183,Aruba,ABW,Latin America & Caribbean,1989,11507.217151,72.531,60443.0,1.874231,9.350730,11.009456,9.408169
4359,Aruba,ABW,Latin America & Caribbean,1990,12187.536361,72.546,62753.0,3.821783,9.408169,11.046962,9.490544


In [8]:
print("Panel year range:", int(panel_events_df["year"].min()), "-", int(panel_events_df["year"].max()))
print("Number of countries:", panel_events_df["country_code"].nunique())

panel_events_df[["country_name", "country_code", "wb_region", "year"]].head()


Panel year range: 1960 - 2023
Number of countries: 214


,country_name,country_code,wb_region,year
3665,Aruba,ABW,Latin America & Caribbean,1986
3833,Aruba,ABW,Latin America & Caribbean,1987
4007,Aruba,ABW,Latin America & Caribbean,1988
4183,Aruba,ABW,Latin America & Caribbean,1989
4359,Aruba,ABW,Latin America & Caribbean,1990


In [9]:
panel_events_df["asian_financial_crisis_9798"] = panel_events_df["year"].isin([1997, 1998]).astype(int)

panel_events_df["global_financial_crisis_0809"] = panel_events_df["year"].isin([2008, 2009]).astype(int)

panel_events_df["covid_shock_2020"] = (panel_events_df["year"] == 2020).astype(int)

panel_events_df["covid_rebound_2021"] = (panel_events_df["year"] == 2021).astype(int)

panel_events_df["ukraine_energy_shock_2022_2024"] = panel_events_df["year"].between(2022, 2024).astype(int)

panel_events_df["high_global_rates_2023_2024"] = panel_events_df["year"].between(2023, 2024).astype(int)

event_cols = [
    "asian_financial_crisis_9798",
    "global_financial_crisis_0809",
    "covid_shock_2020",
    "covid_rebound_2021",
    "ukraine_energy_shock_2022_2024",
    "high_global_rates_2023_2024"
]

panel_events_df[event_cols].head(12)


,asian_financial_crisis_9798,global_financial_crisis_0809,covid_shock_2020,covid_rebound_2021,ukraine_energy_shock_2022_2024,high_global_rates_2023_2024
3665,0,0,0,0,0,0
3833,0,0,0,0,0,0
4007,0,0,0,0,0,0
4183,0,0,0,0,0,0
4359,0,0,0,0,0,0
4551,0,0,0,0,0,0
4744,0,0,0,0,0,0
4938,0,0,0,0,0,0
5133,0,0,0,0,0,0
5329,0,0,0,0,0,0


In [10]:
panel_events_df["asia_crisis_exposed_9798"] = np.where(
    (panel_events_df["asian_financial_crisis_9798"] == 1) &
    (panel_events_df["wb_region"] == "East Asia & Pacific"),
    1,
    0
)

energy_sensitive_regions = [
    "Europe & Central Asia",
    "Middle East, North Africa, Afghanistan & Pakistan"
]

panel_events_df["energy_shock_exposed_2022_2024"] = np.where(
    (panel_events_df["ukraine_energy_shock_2022_2024"] == 1) &
    (panel_events_df["wb_region"].isin(energy_sensitive_regions)),
    1,
    0
)

panel_events_df[[
    "wb_region",
    "year",
    "asian_financial_crisis_9798",
    "asia_crisis_exposed_9798",
    "ukraine_energy_shock_2022_2024",
    "energy_shock_exposed_2022_2024"
]].head(15)


,wb_region,year,asian_financial_crisis_9798,asia_crisis_exposed_9798,ukraine_energy_shock_2022_2024,energy_shock_exposed_2022_2024
3665,Latin America & Caribbean,1986,0,0,0,0
3833,Latin America & Caribbean,1987,0,0,0,0
4007,Latin America & Caribbean,1988,0,0,0,0
4183,Latin America & Caribbean,1989,0,0,0,0
4359,Latin America & Caribbean,1990,0,0,0,0
4551,Latin America & Caribbean,1991,0,0,0,0
4744,Latin America & Caribbean,1992,0,0,0,0
4938,Latin America & Caribbean,1993,0,0,0,0
5133,Latin America & Caribbean,1994,0,0,0,0
5329,Latin America & Caribbean,1995,0,0,0,0


In [11]:
event_summary = pd.DataFrame({
    "event_dummy": event_cols + ["asia_crisis_exposed_9798", "energy_shock_exposed_2022_2024"],
    "count_ones": [
        panel_events_df["asian_financial_crisis_9798"].sum(),
        panel_events_df["global_financial_crisis_0809"].sum(),
        panel_events_df["covid_shock_2020"].sum(),
        panel_events_df["covid_rebound_2021"].sum(),
        panel_events_df["ukraine_energy_shock_2022_2024"].sum(),
        panel_events_df["high_global_rates_2023_2024"].sum(),
        panel_events_df["asia_crisis_exposed_9798"].sum(),
        panel_events_df["energy_shock_exposed_2022_2024"].sum()
    ]
})

event_summary


,event_dummy,count_ones
0,asian_financial_crisis_9798,403
1,global_financial_crisis_0809,424
2,covid_shock_2020,210
3,covid_rebound_2021,210
4,ukraine_energy_shock_2022_2024,412
5,high_global_rates_2023_2024,203
6,asia_crisis_exposed_9798,66
7,energy_shock_exposed_2022_2024,156


In [12]:
year_event_map = (
    panel_events_df.groupby("year")[event_cols + ["asia_crisis_exposed_9798", "energy_shock_exposed_2022_2024"]]
    .max()
    .reset_index()
    .sort_values("year")
)

year_event_map.tail(20)


,year,asian_financial_crisis_9798,global_financial_crisis_0809,covid_shock_2020,covid_rebound_2021,ukraine_energy_shock_2022_2024,high_global_rates_2023_2024,asia_crisis_exposed_9798,energy_shock_exposed_2022_2024
44,2004,0,0,0,0,0,0,0,0
45,2005,0,0,0,0,0,0,0,0
46,2006,0,0,0,0,0,0,0,0
47,2007,0,0,0,0,0,0,0,0
48,2008,0,1,0,0,0,0,0,0
49,2009,0,1,0,0,0,0,0,0
50,2010,0,0,0,0,0,0,0,0
51,2011,0,0,0,0,0,0,0,0
52,2012,0,0,0,0,0,0,0,0
53,2013,0,0,0,0,0,0,0,0


In [13]:
validation_tables = {}

for col in event_cols:
    temp = (
        panel_events_df.groupby(col)["gdp_per_capita_usd"]
        .agg(["mean", "median", "count"])
        .reset_index()
    )
    validation_tables[col] = temp

for name, table in validation_tables.items():
    print(f"\n{name}")
    display(table.round(4))



asian_financial_crisis_9798


,asian_financial_crisis_9798,mean,median,count
0,0,9519.9190,2141.9904,10970
1,1,7945.8415,2067.5981,403



global_financial_crisis_0809


,global_financial_crisis_0809,mean,median,count
0,0,9194.1003,2063.3278,10949
1,1,16437.4569,5215.9459,424



covid_shock_2020


...

,covid_rebound_2021,mean,median,count
0,0,9257.8539,2082.6238,11163
1,1,20429.8178,7268.8995,210



ukraine_energy_shock_2022_2024


,ukraine_energy_shock_2022_2024,mean,median,count
0,0,9011.0985,2025.0816,10961
1,1,21517.0756,7763.7781,412



high_global_rates_2023_2024


,high_global_rates_2023_2024,mean,median,count
0,0,9238.0917,2084.3379,11170
1,1,21902.4676,7826.3538,203


In [14]:
SAVE_EVENTS_PATH = "/Users/tonytony/Final Project/Data/Cleaned/panel_with_event_dummies.csv"

panel_events_df.to_csv(SAVE_EVENTS_PATH, index=False)

print("Saved file:")
print(SAVE_EVENTS_PATH)


Saved file:
/Users/tonytony/Final Project/Data/Cleaned/panel_with_event_dummies.csv


In [19]:
basic_event_features = [
    "asian_financial_crisis_9798",
    "global_financial_crisis_0809",
    "covid_shock_2020",
    "covid_rebound_2021"
]

extended_event_features = [
    "asian_financial_crisis_9798",
    "global_financial_crisis_0809",
    "covid_shock_2020",
    "covid_rebound_2021",
    "ukraine_energy_shock_2022_2024",
    "asia_crisis_exposed_9798",
    "energy_shock_exposed_2022_2024"
]

print("Basic event features:")
print(basic_event_features)

print("\nExtended event features:")
print(extended_event_features)


Basic event features:
['asian_financial_crisis_9798', 'global_financial_crisis_0809', 'covid_shock_2020', 'covid_rebound_2021']

Extended event features:
['asian_financial_crisis_9798', 'global_financial_crisis_0809', 'covid_shock_2020', 'covid_rebound_2021', 'ukraine_energy_shock_2022_2024', 'asia_crisis_exposed_9798', 'energy_shock_exposed_2022_2024']


In [20]:
model_cols = [
    "target_log_gdp_next_year",
    "log_population_total",
    "life_expectancy_years",
    "asian_financial_crisis_9798",
    "global_financial_crisis_0809",
    "covid_shock_2020",
    "covid_rebound_2021",
    "ukraine_energy_shock_2022_2024",
    "wb_region",
    "year"
]

example_model_df = panel_events_df[model_cols].dropna().copy()

example_model_df["target_log_gdp_next_year"] = example_model_df["target_log_gdp_next_year"].astype(float)
example_model_df["log_population_total"] = example_model_df["log_population_total"].astype(float)
example_model_df["life_expectancy_years"] = example_model_df["life_expectancy_years"].astype(float)

example_model_df["asian_financial_crisis_9798"] = example_model_df["asian_financial_crisis_9798"].astype("int64")
example_model_df["global_financial_crisis_0809"] = example_model_df["global_financial_crisis_0809"].astype("int64")
example_model_df["covid_shock_2020"] = example_model_df["covid_shock_2020"].astype("int64")
example_model_df["covid_rebound_2021"] = example_model_df["covid_rebound_2021"].astype("int64")
example_model_df["ukraine_energy_shock_2022_2024"] = example_model_df["ukraine_energy_shock_2022_2024"].astype("int64")

example_model_df["year"] = example_model_df["year"].astype("int64")
example_model_df["wb_region"] = example_model_df["wb_region"].astype(str)

print(example_model_df.dtypes)

example_model = smf.ols(
    formula="""
    target_log_gdp_next_year ~ log_population_total
    + life_expectancy_years
    + asian_financial_crisis_9798
    + global_financial_crisis_0809
    + covid_shock_2020
    + covid_rebound_2021
    + ukraine_energy_shock_2022_2024
    + C(wb_region)
    + C(year)
    """,
    data=example_model_df
).fit(cov_type="HC3")

coef_table = pd.DataFrame({
    "coefficient": example_model.params,
    "p_value": example_model.pvalues
}).round(4)

coef_table.head(20)


target_log_gdp_next_year          float64
log_population_total              float64
life_expectancy_years             float64
asian_financial_crisis_9798         int64
global_financial_crisis_0809        int64
covid_shock_2020                    int64
covid_rebound_2021                  int64
ukraine_energy_shock_2022_2024      int64
wb_region                             str
year                                int64
dtype: object


,coefficient,p_value
Intercept,1.1505,0.0000
C(wb_region)[T.Europe & Central Asia],0.5220,0.0000
C(wb_region)[T.Latin America & Caribbean ],0.0630,0.0107
"C(wb_region)[T.Middle East, North Africa, Afghanistan & Pakistan]",0.3925,0.0000
C(wb_region)[T.North America],1.3814,0.0000
C(wb_region)[T.South Asia],-0.5533,0.0000
C(wb_region)[T.Sub-Saharan Africa ],0.2441,0.0000
C(year)[T.1961],-0.0035,0.9737
C(year)[T.1962],0.0104,0.9218
C(year)[T.1963],0.0356,0.7348


In [23]:
INFLATION_PATH = "/Users/tonytony/Final Project/Data/Raw/inflation.csv"
UNEMPLOYMENT_PATH = "/Users/tonytony/Final Project/Data/Raw/unemployment.csv"
INTERNET_PATH = "/Users/tonytony/Final Project/Data/Raw/individuals_using_ the_Internet.csv"

print("Inflation path:", INFLATION_PATH)
print("Unemployment path:", UNEMPLOYMENT_PATH)
print("Internet path:", INTERNET_PATH)


Inflation path: /Users/tonytony/Final Project/Data/Raw/inflation.csv
Unemployment path: /Users/tonytony/Final Project/Data/Raw/unemployment.csv
Internet path: /Users/tonytony/Final Project/Data/Raw/individuals_using_ the_Internet.csv


In [24]:
inflation_long = add_region_info(to_long(load_wdi(INFLATION_PATH), "inflation_pct"))
unemployment_long = add_region_info(to_long(load_wdi(UNEMPLOYMENT_PATH), "unemployment_pct"))
internet_long = add_region_info(to_long(load_wdi(INTERNET_PATH), "internet_users_pct"))

inflation_long = inflation_long[inflation_long["inflation_pct"].notna()].copy()
unemployment_long = unemployment_long[unemployment_long["unemployment_pct"].notna()].copy()
internet_long = internet_long[internet_long["internet_users_pct"].notna()].copy()

print("Inflation long shape:", inflation_long.shape)
print("Unemployment long shape:", unemployment_long.shape)
print("Internet long shape:", internet_long.shape)


Inflation long shape: (8971, 8)
Unemployment long shape: (6531, 8)
Internet long shape: (6224, 8)


In [25]:
panel_extended_df = panel_events_df.merge(
    inflation_long[["country_code", "year", "inflation_pct"]],
    on=["country_code", "year"],
    how="left"
).merge(
    unemployment_long[["country_code", "year", "unemployment_pct"]],
    on=["country_code", "year"],
    how="left"
).merge(
    internet_long[["country_code", "year", "internet_users_pct"]],
    on=["country_code", "year"],
    how="left"
)

print("Extended panel shape:", panel_extended_df.shape)
panel_extended_df.head()


Extended panel shape: (11373, 22)


,country_name,country_code,wb_region,year,gdp_per_capita_usd,life_expectancy_years,population_total,population_growth_pct,log_gdp_per_capita,log_population_total,...,global_financial_crisis_0809,covid_shock_2020,covid_rebound_2021,ukraine_energy_shock_2022_2024,high_global_rates_2023_2024,asia_crisis_exposed_9798,energy_shock_exposed_2022_2024,inflation_pct,unemployment_pct,internet_users_pct
0,Aruba,ABW,Latin America & Caribbean,1986,6767.559229,71.831,59931.0,-2.911159,8.819896,11.000949,...,0,0,0,0,0,0,0,1.073966,NaN,NaN
1,Aruba,ABW,Latin America & Caribbean,1987,8244.045660,72.448,59159.0,-1.288148,9.017246,10.987984,...,0,0,0,0,0,0,0,3.643045,NaN,NaN
2,Aruba,ABW,Latin America & Caribbean,1988,10056.261393,72.519,59331.0,0.290742,9.215951,10.990887,...,0,0,0,0,0,0,0,3.121868,NaN,NaN
3,Aruba,ABW,Latin America & Caribbean,1989,11507.217151,72.531,60443.0,1.874231,9.350730,11.009456,...,0,0,0,0,0,0,0,3.991628,NaN,NaN
4,Aruba,ABW,Latin America & Caribbean,1990,12187.536361,72.546,62753.0,3.821783,9.408169,11.046962,...,0,0,0,0,0,0,0,5.836688,NaN,0.0


In [26]:
new_var_summary = pd.DataFrame({
    "variable": ["inflation_pct", "unemployment_pct", "internet_users_pct"],
    "missing_count": [
        panel_extended_df["inflation_pct"].isna().sum(),
        panel_extended_df["unemployment_pct"].isna().sum(),
        panel_extended_df["internet_users_pct"].isna().sum()
    ],
    "missing_ratio_pct": [
        round(panel_extended_df["inflation_pct"].isna().mean() * 100, 2),
        round(panel_extended_df["unemployment_pct"].isna().mean() * 100, 2),
        round(panel_extended_df["internet_users_pct"].isna().mean() * 100, 2)
    ]
})

new_var_summary


,variable,missing_count,missing_ratio_pct
0,inflation_pct,2720,23.92
1,unemployment_pct,5350,47.04
2,internet_users_pct,5430,47.74


In [27]:
panel_extended_df["inflation_pct_clean"] = panel_extended_df["inflation_pct"].copy()

infl_q01 = panel_extended_df["inflation_pct_clean"].quantile(0.01)
infl_q99 = panel_extended_df["inflation_pct_clean"].quantile(0.99)

panel_extended_df["inflation_pct_clean"] = panel_extended_df["inflation_pct_clean"].clip(lower=infl_q01, upper=infl_q99)

panel_extended_df["unemployment_pct_clean"] = panel_extended_df["unemployment_pct"].where(
    panel_extended_df["unemployment_pct"].between(0, 100),
    np.nan
)

panel_extended_df["internet_users_pct_clean"] = panel_extended_df["internet_users_pct"].where(
    panel_extended_df["internet_users_pct"].between(0, 100),
    np.nan
)

panel_extended_df[[
    "inflation_pct",
    "inflation_pct_clean",
    "unemployment_pct",
    "unemployment_pct_clean",
    "internet_users_pct",
    "internet_users_pct_clean"
]].describe().T.round(4)


,count,mean,std,min,25%,50%,75%,max
inflation_pct,8653.0,22.3397,323.8285,-17.6404,2.1480,4.8694,10.0695,23773.1318
inflation_pct_clean,8653.0,10.5600,22.0845,-2.8285,2.1480,4.8694,10.0695,173.4835
unemployment_pct,6023.0,8.1315,6.0986,0.1000,3.6830,6.3630,11.0015,38.8000
unemployment_pct_clean,6023.0,8.1315,6.0986,0.1000,3.6830,6.3630,11.0015,38.8000
internet_users_pct,5943.0,30.8916,32.0796,0.0000,1.6882,17.6600,59.3850,100.0000
internet_users_pct_clean,5943.0,30.8916,32.0796,0.0000,1.6882,17.6600,59.3850,100.0000


In [28]:
model_cols = [
    "target_log_gdp_next_year",
    "log_population_total",
    "life_expectancy_years",
    "population_growth_pct",
    "inflation_pct_clean",
    "unemployment_pct_clean",
    "internet_users_pct_clean",
    "asian_financial_crisis_9798",
    "global_financial_crisis_0809",
    "covid_shock_2020",
    "covid_rebound_2021",
    "ukraine_energy_shock_2022_2024",
    "wb_region",
    "year"
]

extended_model_df = panel_extended_df[model_cols].dropna().copy()

extended_model_df["target_log_gdp_next_year"] = extended_model_df["target_log_gdp_next_year"].astype(float)
extended_model_df["log_population_total"] = extended_model_df["log_population_total"].astype(float)
extended_model_df["life_expectancy_years"] = extended_model_df["life_expectancy_years"].astype(float)
extended_model_df["population_growth_pct"] = extended_model_df["population_growth_pct"].astype(float)
extended_model_df["inflation_pct_clean"] = extended_model_df["inflation_pct_clean"].astype(float)
extended_model_df["unemployment_pct_clean"] = extended_model_df["unemployment_pct_clean"].astype(float)
extended_model_df["internet_users_pct_clean"] = extended_model_df["internet_users_pct_clean"].astype(float)

dummy_cols = [
    "asian_financial_crisis_9798",
    "global_financial_crisis_0809",
    "covid_shock_2020",
    "covid_rebound_2021",
    "ukraine_energy_shock_2022_2024"
]

for col in dummy_cols:
    extended_model_df[col] = extended_model_df[col].astype("int64")

extended_model_df["year"] = extended_model_df["year"].astype("int64")
extended_model_df["wb_region"] = extended_model_df["wb_region"].astype(str)

print("Extended model dataset shape:", extended_model_df.shape)
extended_model_df.dtypes


Extended model dataset shape: (4722, 14)


target_log_gdp_next_year          float64
log_population_total              float64
life_expectancy_years             float64
population_growth_pct             float64
inflation_pct_clean               float64
unemployment_pct_clean            float64
internet_users_pct_clean          float64
asian_financial_crisis_9798         int64
global_financial_crisis_0809        int64
covid_shock_2020                    int64
covid_rebound_2021                  int64
ukraine_energy_shock_2022_2024      int64
wb_region                             str
year                                int64
dtype: object

In [29]:
core_model_same_sample = smf.ols(
    formula="""
    target_log_gdp_next_year ~ log_population_total
    + life_expectancy_years
    + asian_financial_crisis_9798
    + global_financial_crisis_0809
    + covid_shock_2020
    + covid_rebound_2021
    + ukraine_energy_shock_2022_2024
    + C(wb_region)
    + C(year)
    """,
    data=extended_model_df
).fit(cov_type="HC3")

print("Core model fitted successfully")
print("R-squared:", round(core_model_same_sample.rsquared, 4))
print("Adj. R-squared:", round(core_model_same_sample.rsquared_adj, 4))


Core model fitted successfully
R-squared: 0.7441
Adj. R-squared: 0.742


In [30]:
extended_gdp_model = smf.ols(
    formula="""
    target_log_gdp_next_year ~ log_population_total
    + life_expectancy_years
    + population_growth_pct
    + inflation_pct_clean
    + unemployment_pct_clean
    + internet_users_pct_clean
    + asian_financial_crisis_9798
    + global_financial_crisis_0809
    + covid_shock_2020
    + covid_rebound_2021
    + ukraine_energy_shock_2022_2024
    + C(wb_region)
    + C(year)
    """,
    data=extended_model_df
).fit(cov_type="HC3")

print("Extended model fitted successfully")
print("R-squared:", round(extended_gdp_model.rsquared, 4))
print("Adj. R-squared:", round(extended_gdp_model.rsquared_adj, 4))


Extended model fitted successfully
R-squared: 0.8092
Adj. R-squared: 0.8075


In [31]:
model_compare = pd.DataFrame([
    {
        "Model": "Core model",
        "R_squared": core_model_same_sample.rsquared,
        "Adj_R_squared": core_model_same_sample.rsquared_adj,
        "AIC": core_model_same_sample.aic,
        "BIC": core_model_same_sample.bic,
        "N_obs": int(core_model_same_sample.nobs)
    },
    {
        "Model": "Extended model",
        "R_squared": extended_gdp_model.rsquared,
        "Adj_R_squared": extended_gdp_model.rsquared_adj,
        "AIC": extended_gdp_model.aic,
        "BIC": extended_gdp_model.bic,
        "N_obs": int(extended_gdp_model.nobs)
    }
])

model_compare.round(4)


,Model,R_squared,Adj_R_squared,AIC,BIC,N_obs
0,Core model,0.7441,0.7420,11172.3383,11430.7378,4722
1,Extended model,0.8092,0.8075,9793.5812,10077.8206,4722


In [32]:
main_vars = [
    "log_population_total",
    "life_expectancy_years",
    "population_growth_pct",
    "inflation_pct_clean",
    "unemployment_pct_clean",
    "internet_users_pct_clean",
    "asian_financial_crisis_9798",
    "global_financial_crisis_0809",
    "covid_shock_2020",
    "covid_rebound_2021",
    "ukraine_energy_shock_2022_2024"
]

coef_table_extended = pd.DataFrame({
    "variable": main_vars,
    "coefficient": [extended_gdp_model.params.get(v, np.nan) for v in main_vars],
    "p_value": [extended_gdp_model.pvalues.get(v, np.nan) for v in main_vars],
    "ci_lower": [extended_gdp_model.conf_int().loc[v, 0] if v in extended_gdp_model.params.index else np.nan for v in main_vars],
    "ci_upper": [extended_gdp_model.conf_int().loc[v, 1] if v in extended_gdp_model.params.index else np.nan for v in main_vars]
}).round(4)

coef_table_extended


,variable,coefficient,p_value,ci_lower,ci_upper
0,log_population_total,-0.0578,0.0000,-0.0689,-0.0468
1,life_expectancy_years,0.1075,0.0000,0.1004,0.1146
2,population_growth_pct,0.0480,0.0000,0.0289,0.0672
3,inflation_pct_clean,-0.0058,0.0000,-0.0072,-0.0044
4,unemployment_pct_clean,0.0084,0.0003,0.0038,0.0129
5,internet_users_pct_clean,0.0245,0.0000,0.0231,0.0258
6,asian_financial_crisis_9798,-0.5977,0.0000,-0.7674,-0.4280
7,global_financial_crisis_0809,-0.7344,0.0000,-0.9017,-0.5671
8,covid_shock_2020,-0.9515,0.0000,-1.0868,-0.8162
9,covid_rebound_2021,-0.9076,0.0000,-1.0444,-0.7708


In [33]:
SAVE_EXTENDED_PANEL_PATH = "/Users/tonytony/Final Project/Data/Cleaned/panel_with_event_dummies_and_extra_drivers.csv"

panel_extended_df.to_csv(SAVE_EXTENDED_PANEL_PATH, index=False)

print("Saved file:")
print(SAVE_EXTENDED_PANEL_PATH)


Saved file:
/Users/tonytony/Final Project/Data/Cleaned/panel_with_event_dummies_and_extra_drivers.csv
